# 03 · Churn diagnostic for TANL's negative-label selection

Measures how TANL's set of M = 1,000 selected negatives evolves over the test stream, **without changing
anything TANL computes** (checked in step 1: scores with and without the tracker are bit-identical, so the
diagnostic figures and the headline numbers come from the same run).

Per batch: **churn** (1 − overlap with the previous batch's set) · **survival** of the first-batch set and
of the initial text/noise set · **overlap with the oracle** set (top-M by the *true* OOD−ID activation
difference of the stream) · distinct labels used so far · **queue purity** (true-ID share of the ID
queue, true-OOD share of the OOD queue, by mirroring the FIFOs with ground truth) · admission precision ·
the automatic threshold. Per run: dwell times and batches to reach 90 % of the oracle-overlap plateau;
for temporal-shift streams the same after every shift.

**Decision rule (plan):** final survival of the first-batch selection **> 50 %** supports the cold-start
framing; **< 10 %** means switching to a shift-only framing before writing.

**Settings:** GPU recommended (≈ 20–30 min). **Inputs:** `oodlab-code` + notebook 01's output.

In [ ]:
# ── Set-up 1/2: find the oodlab code (attached as a private Kaggle dataset) ─────────
import os, sys

def find_oodlab(roots, max_depth=4):
    """Bounded breadth-first search for a folder containing oodlab/__init__.py.
    Never walks image folders (the old notebook hung on a recursive glob over ImageNet)."""
    skip = {"ILSVRC", "train", "val", "test", "images", "__pycache__", "cache", "results", "logs"}
    for root in [r for r in roots if r and os.path.isdir(r)]:
        frontier = [(root, 0)]
        while frontier:
            d, depth = frontier.pop(0)
            if os.path.isfile(os.path.join(d, "oodlab", "__init__.py")):
                return d
            if depth < max_depth:
                try:
                    subs = sorted(e.path for e in os.scandir(d)
                                  if e.is_dir(follow_symlinks=False) and e.name not in skip)
                except OSError:
                    continue
                frontier += [(s, depth + 1) for s in subs]
    return None

ROOTS = [os.environ.get("OODLAB_CODE"), "/kaggle/input", "/kaggle/working", os.getcwd()]
CODE_ROOT = find_oodlab(ROOTS)
if CODE_ROOT is None:   # the dataset was uploaded as a zip that Kaggle did not unpack -> unpack it ourselves
    import zipfile
    for root in [r for r in ROOTS[:2] if r and os.path.isdir(r)]:
        for dp, dn, fns in os.walk(root):
            dn[:] = [d for d in dn if d not in {"ILSVRC", "train", "val", "test", "images"}] if dp.count(os.sep) - root.count(os.sep) < 3 else []
            for fn in fns:
                if fn.startswith("oodlab") and fn.endswith(".zip"):
                    zipfile.ZipFile(os.path.join(dp, fn)).extractall("/kaggle/working/_oodlab_code")
    CODE_ROOT = find_oodlab(["/kaggle/working/_oodlab_code"])
if CODE_ROOT is None:
    raise FileNotFoundError("oodlab code not found: attach your private dataset with the code "
                            "(Add Input -> Your Datasets -> oodlab-code). See GUIDE.md, section 2.")
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)
import oodlab
print("oodlab", oodlab.__version__, "loaded from", CODE_ROOT)

In [ ]:
# ── Set-up 2/2: packages, folders, logging ─────────────────────────────────────────
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True, gdown=False)      # CLIP ships inside oodlab; this only adds small pip packages if missing
ctx = start("03_churn_diagnostic", need_gpu=False)
log = ctx.log                                   # log.info(...) -> screen + /kaggle/working/logs/03_churn_diagnostic.log
import numpy as np, pandas as pd, torch, time, json
from oodlab.report import show, save_tables, write_report

In [ ]:
from oodlab.methods import TANL, TANLConfig
from oodlab.runner import Runner, run_stream, summarize, seed_method
from oodlab.stream import pair_stream, temporal_stream, TEMPORAL_ORDERS
from oodlab.diagnostics import ChurnTracker, oracle_selection, assert_passive, plot_run, plot_dwell, decision
from oodlab.report import find_cache

from oodlab.report import best_tanl_config
BACKBONE = "ViT-B/16"
CONFIG_NAME = "auto"   # "paper" | "official_sh" | "auto" (= the one closest to 9.81 in notebook 02, if attached)
PRECISION = "fp16" if ctx.device == "cuda" else "fp32"
if CONFIG_NAME == "auto":
    prev = oodlab.paths.find_file("tanl_summary.csv", [ctx.dirs.results, ctx.dirs.inputs], max_depth=5)
    CONFIG_NAME = (best_tanl_config(prev, precision=PRECISION) if prev else None) or "paper"
CONFIG = {"paper": TANLConfig.paper(), "official_sh": TANLConfig.official_sh()}[CONFIG_NAME]
log.info(f"TANL configuration for the diagnostic: {CONFIG_NAME} ({PRECISION})")
SEEDS = [0, 1, 2]
DEFAULT_STREAMS = [("imagenet_val_all", ["inaturalist", "sun", "places", "textures_all"]),   # Four-OOD
                   ("imagenet_test", ["ssb_hard", "ninco"])]                                  # OpenOOD near
RUN_TEMPORAL = True
if ctx.smoke:
    CONFIG, SEEDS = CONFIG.with_(num_neg=100), [0, 1]

cache = find_cache(ctx, BACKBONE)
bank = cache.load_textbank()
runner = Runner(cache, device=ctx.device, logger=log)

def make(record):
    c = CONFIG.with_(logit_scale=bank.logit_scale, record=record)
    if PRECISION == "fp16":
        c = c.with_(emulate_fp16=True, exact_fp16_final=True)
    return TANL(bank.id_text, bank.corpus_text, bank.noise_feats, c, device=ctx.device, n_neglabel=bank.n_selected)
os.makedirs(ctx.path("plots", "x"), exist_ok=True)

### 1 · The tracker is passive

In [ ]:
pairs = [(i, o) for i, oods in DEFAULT_STREAMS for o in oods if cache.has(i) and cache.has(o)]
if not pairs:
    raise RuntimeError(f"no ID/OOD pair in the cache ({cache.available()}); finish notebook 01 first")
s = pair_stream(runner.get(pairs[0][0]), runner.get(pairs[0][1]), seed=0).to(ctx.device)
print(assert_passive(make(False), make(True), s, device=ctx.device))
del s

### 2 · Default (shuffled) streams, one reset per ID/OOD pair

In [ ]:
rows, frames = [], {}
for id_name, oods in DEFAULT_STREAMS:
    for ood_name in oods:
        if any(n not in cache.available() for n in [id_name, ood_name]):
            log.warning(f"skip {ood_name}: not cached"); continue
        for seed in SEEDS:
            s = pair_stream(runner.get(id_name), runner.get(ood_name), seed=seed).to(ctx.device)
            m = make(True); seed_method(m, seed); m.reset()
            tr = ChurnTracker({0: oracle_selection(m, s, 0, device=ctx.device)}, name=f"{ood_name}/s{seed}")
            res = run_stream(m, s, ctx.device, hooks=[tr])
            met = res.segment_metrics(0)
            summ = tr.summary()
            rows.append({"id": id_name, "ood": ood_name, "seed": seed, "fpr95": met["fpr95"], "auroc": met["auroc"],
                         **{k: v for k, v in summ.items() if k not in ("name", "shifts", "decision")}})
            frames[(ood_name, seed)] = tr.frame()
            tr.frame().to_csv(ctx.path("per_batch", f"{ood_name}_seed{seed}.csv"), index=False)
            if seed == SEEDS[0]:
                plot_run(tr.frame(), f"TANL {PRECISION}: {id_name} + {ood_name} (seed {seed})",
                         ctx.path("plots", f"churn_{ood_name}.png"))
                plot_dwell(tr.dwell_times(), f"dwell times: {ood_name}", ctx.path("plots", f"dwell_{ood_name}.png"))
            log.info(f"{ood_name} seed {seed}: final survival {summ['final_surv_first']:.3f}, "
                     f"churn(first10) {summ['churn_first10']:.3f}, FPR95 {met['fpr95']:.2f}")
            del s
churn = pd.DataFrame(rows)
per_set = churn.groupby("ood").mean(numeric_only=True).drop(columns=["seed"]).reset_index()
show(per_set, "churn summary (mean over seeds)", cols=["ood", "fpr95", "final_surv_first", "final_surv_init",
     "final_oracle_overlap", "batches_to_90pct_oracle_plateau", "churn_first10", "churn_second_half",
     "unique_labels", "dwell_median", "final_pos_purity", "final_neg_purity"], digits=3)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for (name, seed), f in frames.items():
    if seed != SEEDS[0]:
        continue
    ax[0].plot(f.batch, f.surv_first, label=name); ax[1].plot(f.batch, f.churn, label=name)
    ax[2].plot(f.batch, f.oracle_overlap, label=name)
for a, t in zip(ax, ["survival of first-batch set", "churn", "overlap with oracle set"]):
    a.set_title(t); a.set_xlabel("batch"); a.legend(fontsize=7)
ax[0].axhline(0.5, ls="--", c="gray"); ax[0].axhline(0.1, ls=":", c="gray")
fig.tight_layout(); fig.savefig(ctx.path("plots", "overview.png"), dpi=110); plt.show()

### 3 · Temporal-shift orders (no reset between OOD sets)

In [ ]:
shift_rows = []
if RUN_TEMPORAL and all(n in cache.available() for n in ["imagenet_val_all", *TEMPORAL_ORDERS["I-S-P-T"]]):
    for oname, names in TEMPORAL_ORDERS.items():
        for seed in SEEDS[:1] if ctx.smoke else SEEDS:
            ts = temporal_stream(runner.get("imagenet_val_all"), [runner.get(n) for n in names], seed=seed).to(ctx.device)
            m = make(True); seed_method(m, seed); m.reset()
            orc = {k: oracle_selection(m, ts, k, device=ctx.device) for k in range(len(ts.segments))}
            tr = ChurnTracker(orc, name=f"{oname}/s{seed}")
            res = run_stream(m, ts, ctx.device, hooks=[tr])
            summ = tr.summary()
            for k, seg in enumerate(res.segments):
                met, win = res.segment_metrics(k), res.window_metrics(k, runner.window)
                sh = next((x for x in summ.get("shifts", []) if x["segment"] == k), {})
                shift_rows.append({"order": oname, "seed": seed, "position": k, "ood": seg.name,
                                   "fpr95": met["fpr95"], "post_shift_fpr95": win["fpr95"],
                                   "churn_at_shift": sh.get("churn_at_shift"), "overlap_at_shift": sh.get("overlap_at_shift"),
                                   "batches_to_90pct": sh.get("batches_to_90pct_plateau", summ["batches_to_90pct_oracle_plateau"] if k == 0 else None),
                                   "final_overlap": sh.get("final_overlap", summ["final_oracle_overlap"] if k == 0 else None)})
            if seed == SEEDS[0]:
                f = tr.frame()
                f.to_csv(ctx.path("per_batch", f"temporal_{oname}.csv"), index=False)
                bounds = [int(f[f.segment == k].batch.min()) for k in range(1, len(ts.segments))]
                plot_run(f, f"temporal {oname} (seed {seed})", ctx.path("plots", f"temporal_{oname}.png"), boundaries=bounds)
            del ts
shifts = pd.DataFrame(shift_rows)
if len(shifts):
    show(shifts.groupby(["order", "position", "ood"]).mean(numeric_only=True).drop(columns=["seed"]).reset_index(),
         "post-shift behaviour (mean over seeds)", digits=3)

### 4 · Decision

In [ ]:
surv = float(churn["final_surv_first"].mean()) if len(churn) else float("nan")
verdict = decision(surv)
text = (f"Mean final survival of the first-batch selection over {len(churn)} default streams: **{surv:.1%}** "
        f"(per set: " + ", ".join(f"{r.ood} {r.final_surv_first:.1%}" for r in per_set.itertuples()) + ").\n\n"
        f"Rule: > 50 % supports the cold-start framing; < 10 % means a shift-only framing.\n\n**Verdict: {verdict}**")
print(text.replace("**", ""))
save_tables(ctx, churn_runs=churn, churn_per_set=per_set, churn_shifts=shifts)
write_report(ctx, "TANL churn diagnostic", [("Decision", text), ("Per OOD set (mean over seeds)", per_set),
                                           ("Temporal shift", shifts if len(shifts) else "not run")])
finish(ctx, {"mean_final_survival": surv, "verdict": verdict})